# 内存控制与分块处理

学习目标：估计表格内存，按用途选择列与类型，并用可合并统计量完成 CSV 分块汇总，辨认需要跨块协调的任务。

前置知识：dtype、聚合、缺失值、CSV 读写、循环与函数。

运行环境：Python 3.12、pandas 3.0、PyArrow 25.0。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例均使用自制数据，后续单元沿用 pd 及已导入的标准库名称。当前环境的默认 str 使用 pyarrow 存储；CSV 示例显式使用 C 解析引擎。临时 CSV 在各自的 TemporaryDirectory 上下文结束时清理，最大实验规模为 20,000 行。

## 1 查看表格内存

要汇总销量，只需要销量列；决定如何减少内存前，先查看每列保存了什么。下面的小表包含地区、销量与备注，销量单位为件。

In [1]:
import pandas as pd

sales = pd.DataFrame({
    "region": ["east", "west"] * 4,
    "units": [2, 4, 6, 8] * 2,
    "note": ["daily record"] * 8,
})
print(sales.head(4))  # 表中四种销量模式重复两次。
print(sales["units"].sum())  # 共 40 件。

  region  units          note
0   east      2  daily record
1   west      4  daily record
2   east      6  daily record
3   west      8  daily record
40


info 展示列类型、非缺失计数与内存摘要；memory_usage 返回各列的字节估计，默认包含索引。deep=True 会深入统计 object 列引用的对象，适合检查包含 Python 对象的数据。

这些数值描述表格及索引的存储估计，不是整个 Python 进程的驻留内存，也不是执行期间的峰值；解析缓冲区、其他变量与临时中间结果不在这张表的统计范围内。

In [2]:
sales.info(memory_usage="deep")  # 8 行、3 列，均无缺失；两列 str、一列 int64。
column_bytes = sales.memory_usage(index=True, deep=True)
print(column_bytes)  # Index 单独列出，其余按列名列出字节数。
print("表内存估计（字节）:", column_bytes.sum())
print("不含索引（字节）:", sales.memory_usage(index=False, deep=True).sum())
# 两个总量之差是这里报告的 Index 内存；不是操作的峰值差。

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   region  8 non-null      str  
 1   units   8 non-null      int64
 2   note    8 non-null      str  
dtypes: int64(1), str(2)
memory usage: 452.0 bytes
Index     132
region     96
units      64
note      160
dtype: int64
表内存估计（字节）: 452
不含索引（字节）: 320


### 1.1 字符串后端会影响估计

pandas 3 默认推断出的文本列是 str，安装 PyArrow 时默认使用 Arrow 存储。显式 object 则保存 Python 对象，不能沿用它的内存数字推断默认 str 的占用。

下面对相同文本比较两种表示。deep=True 对 object 与默认 str 的影响可能不同；具体数值随表示和环境变化。

In [3]:
text = pd.Series(["east", "west"] * 100)
python_objects = text.astype(object)
memory = pd.DataFrame({
    "shallow_bytes": [text.memory_usage(), python_objects.memory_usage()],
    "deep_bytes": [text.memory_usage(deep=True), python_objects.memory_usage(deep=True)],
}, index=["str", "object"])
print(text.dtype, text.dtype.storage)  # str pyarrow。
print(memory)
# 当前 str 两种估计相同；object 的 deep 估计还计入字符串对象。
print(text.tolist() == python_objects.tolist())  # True；值和顺序一致。

str pyarrow
        shallow_bytes  deep_bytes
str              2532        2532
object           1732       10732
True


## 2 少读列并选择类型

### 2.1 读取时裁剪

如果任务只需要销量，在 read_csv 中指定 usecols，比先读进全部列再删除更直接。dtype 可以按输入约定指定类型，减少推断差异。下面销量均为无缺失的小整数，用 int16 表示，并核对读取前后的值。

本单元先在临时目录写入 sales，再完整读取与裁剪读取。to_csv 完成写入后关闭文件；退出临时目录上下文会删除 CSV 和目录。后续示例不依赖该文件。

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    csv_path = Path(directory) / "sales.csv"
    sales.to_csv(csv_path, index=False, encoding="utf-8")
    all_columns = pd.read_csv(csv_path, engine="c", encoding="utf-8")
    selected = pd.read_csv(
        csv_path, usecols=["units"], dtype={"units": "int16"},
        engine="c", encoding="utf-8",
    )
    print(selected)  # 8 行 1 列，units 顺序仍为 2、4、6、8，重复两次。
    print(all_columns.shape, selected.shape, selected["units"].dtype)
    print(all_columns.memory_usage(deep=True).sum(), selected.memory_usage(deep=True).sum())
    print(selected["units"].tolist() == sales["units"].tolist())  # True。
print(csv_path.exists())  # False：临时文件已清理。

   units
0      2
1      4
2      6
3      8
4      2
5      4
6      6
7      8
(8, 3) (8, 1) int16
452 148
True
False


### 2.2 类型必须容纳输入

较窄的整数类型减少每个值的存储宽度，但前提是输入范围合适。to_numeric 的 downcast 根据实际值寻找可以容纳数据的较小类型；它不保证每次输入都得到同一宽度。

需要缺失值时还要选择可空类型；降低浮点宽度则需另行检查精度。不能只看某个类型占用少，就忽略输入与输出要求。

In [5]:
small_counts = pd.Series([0, 20, 200])
larger_counts = pd.Series([0, 20, 300])
compact = pd.to_numeric(small_counts, downcast="unsigned")
wider = pd.to_numeric(larger_counts, downcast="unsigned")
nullable = pd.to_numeric(pd.Series([0, None, 200], dtype="Int64"), downcast="unsigned")
print(compact.tolist(), compact.dtype)  # [0, 20, 200]，uint8。
print(wider.tolist(), wider.dtype)  # [0, 20, 300]，uint16；输入变化后宽度不同。
print(nullable.tolist(), nullable.dtype)  # [0, <NA>, 200]，UInt8。
print(small_counts.memory_usage(index=False), compact.memory_usage(index=False))
# 这里的三个 int64 值占 24 字节，三个 uint8 值占 3 字节，不含索引。

[0, 20, 200] uint8
[0, 20, 300] uint16
[0, <NA>, 200] UInt8
24 3


## 3 分类压缩与不同值数量

category 保存类别表和每行的整数编码。大量重复、不同值较少的列适合考虑这种表示；如果每一行几乎都不同，类别表和编码可能使内存增加。

下面两列都有 1,000 行，一列只有两种地区，另一列每个编号都不同。比较时保留相同的行索引，并将转换结果还原为 str 核对值。

In [6]:
repeated = pd.Series(["east", "west"] * 500)
unique_ids = pd.Series([f"ID{i:04d}" for i in range(1000)])
rows = []
for name, values in [("two_regions", repeated), ("unique_ids", unique_ids)]:
    categorical = values.astype("category")
    rows.append({
        "column": name,
        "unique_count": values.nunique(),
        "str_bytes": values.memory_usage(deep=True),
        "category_bytes": categorical.memory_usage(deep=True),
    })
    print(name, categorical.astype("str").equals(values))  # 两次均为 True。
print(pd.DataFrame(rows).set_index("column"))
# 当前环境：地区列转 category 后估计减少，唯一编号列反而增加。

two_regions True
unique_ids True
             unique_count  str_bytes  category_bytes
column                                              
two_regions             2      12132            1157
unique_ids           1000      14132           16257


## 4 分块读取与累计

### 4.1 每次只处理一块

read_csv 指定 chunksize 后返回可迭代的 TextFileReader，每次得到最多相应行数的 DataFrame，最后一块可能较短。用 with 管理读取器，退出时关闭，再清理临时目录。low_memory 是解析器内部选项，不能替代 chunksize 提供逐块结果。

下面计算金额均值，单位为分，缺失表示没有有效金额。若各块长度和缺失数量不同，平均各块均值会用错权重；需要累积的是总和与非缺失计数。

![三块分别贡献总和 30、0、90 和有效计数 2、0、1；合并为 120 除以 3，得到均值 40 分。](image/illustration/21-01-chunked-mean.svg)

图中全缺失块贡献计数 0，不增加均值分母。这里只讨论可由总和与计数合并的均值，不暗示中位数或全局去重也能使用相同摘要。

下面用同样的三块数据逐块更新两个累计量；只保留三行过程摘要，不保存各块原表。对照图检查 sum、count 和最终分母。

In [7]:
csv_text = "id,amount_cents\n1,10\n2,20\n3,\n4,\n5,90\n"
# 只累计总额和有效数量，不保存各块原始数据，也不平均各块均值。
total_sum = 0
total_count = 0
chunk_rows = []
with TemporaryDirectory() as directory:
    csv_path = Path(directory) / "amounts.csv"
    csv_path.write_text(csv_text, encoding="utf-8")
    with pd.read_csv(
        csv_path, chunksize=2, usecols=["amount_cents"],
        dtype={"amount_cents": "Int64"}, engine="c", encoding="utf-8",
    ) as reader:
        for number, chunk in enumerate(reader, start=1):
            values = chunk["amount_cents"]
            # count 排除缺失值；全缺失块贡献数量 0，不能按块行数作分母。
            part_sum = int(values.sum())
            part_count = int(values.count())
            total_sum += part_sum
            total_count += part_count
            chunk_rows.append([number, len(chunk), part_sum, part_count])
# 上面只保留每块的四个摘要，读取器和临时文件在 with 结束时关闭。
chunk_stats = pd.DataFrame(chunk_rows, columns=["chunk", "rows", "sum", "count"])
mean_cents = total_sum / total_count  # 本例共有 3 个有效值。
print(chunk_stats)  # 行数为 2、2、1；sum 为 30、0、90；count 为 2、0、1。
print(total_sum, total_count, mean_cents)  # 120，3，40.0 分。
print(csv_path.exists())  # False：读取器已退出，临时文件已清理。

   chunk  rows  sum  count
0      1     2   30      2
1      2     2    0      0
2      3     1   90      1
120 3 40.0
False


### 4.2 块均值不能直接平均

整体均值等于所有有效值之和除以有效值总数。若使用块均值，需要按每块有效值的数量加权；块的行数包含缺失，不能替代有效计数。

继续使用 chunk_stats。第一块的均值 15 代表两条有效观测，最后一块的均值 90 只代表一条。直接平均这两个数会把较短的块赋予过大的权重。计数为零的块不参与均值权重；累计 sum 和 count 可以避免处理这个块的未定义均值。

In [8]:
valid_stats = chunk_stats.loc[chunk_stats["count"] > 0].copy()
valid_stats["mean"] = valid_stats["sum"] / valid_stats["count"]
naive_mean = valid_stats["mean"].mean()
weighted_mean = (valid_stats["mean"] * valid_stats["count"]).sum() / valid_stats["count"].sum()
print(valid_stats)  # 第 1 块均值为 15，第 3 块均值为 90。
print(naive_mean, weighted_mean)  # 52.5 与 40.0；整体有效观测均值是 40.0。

   chunk  rows  sum  count  mean
0      1     2   30      2  15.0
2      3     1   90      1  90.0


52.5 40.0


### 4.3 没有有效值的边界

sum 默认跳过缺失，对空序列或全缺失序列返回 0；count 返回 0。这里用 0 作为累计和的起点，不代表实际观测到一个零金额。

如果最终有效计数仍为 0，本章约定均值输出 pd.NA，避免除以零。若业务还要求“没有有效值时总额也必须缺失”，可在最后依据总计数处理，或对一次性求和使用 min_count=1。

In [9]:
for name, values in [
    ("all_missing", pd.Series([None, None], dtype="Int64")),
    ("empty", pd.Series([], dtype="Int64")),
]:
    count = int(values.count())
    total = int(values.sum())
    mean = total / count if count else pd.NA
    print(name, total, count, mean, values.sum(min_count=1))
# 两次均为：累计和 0、有效计数 0、均值 <NA>、min_count=1 的和 <NA>。

all_missing 0 0 <NA> <NA>
empty 0 0 <NA> <NA>


## 5 需要跨块协调的任务

### 5.1 跨块重复

每块内去重后，重复键仍可能出现在另一块。下面把相同 id=2 放在两个块中，分别去重无法发现这对重复。

小表可以合并后统一处理。大表若维护“已见过的键”，该状态也会随不同键的数量增长；分块并不自动限制所有状态的大小。

In [10]:
first = pd.DataFrame({"id": [1, 2]})
second = pd.DataFrame({"id": [2, 3]})
local = pd.concat([first.drop_duplicates("id"), second.drop_duplicates("id")], ignore_index=True)
global_unique = local.drop_duplicates("id", keep="first", ignore_index=True)
print(local["id"].tolist())  # [1, 2, 2, 3]，跨块重复仍在。
print(global_unique["id"].tolist())  # [1, 2, 3]。
print(local.shape, global_unique.shape)  # (4, 1) 与 (3, 1)。

[1, 2, 2, 3]
[1, 2, 3]
(4, 1) (3, 1)


### 5.2 全局排序

各块内部有序，不代表按读取顺序拼接后全局有序。全局排序需要比较块与块之间的键，不能只调用每块的 sort_values 就结束。

In [11]:
first = pd.DataFrame({"value": [3, 1]})
second = pd.DataFrame({"value": [4, 2]})
locally_sorted = pd.concat([first.sort_values("value"), second.sort_values("value")], ignore_index=True)
print(locally_sorted["value"].tolist())  # [1, 3, 2, 4]，不是全局排序。
globally_sorted = locally_sorted.sort_values("value", ignore_index=True)
print(globally_sorted["value"].tolist())  # [1, 2, 3, 4]。

[1, 3, 2, 4]
[1, 2, 3, 4]


### 5.3 连接对象不能随意分块配对

连接依赖键关系，不依赖文件中的块编号。把左右两表的第一个块互相连接、第二个块互相连接，可能漏掉位于不同块的匹配键。

如果右侧是一张能放入内存的完整小表，可以让每个左块与它连接；两侧都很大时，需要按键分区等协调方法，或使用能处理这类任务的工具。

In [12]:
left_parts = [pd.DataFrame({"id": [1]}), pd.DataFrame({"id": [2]})]
right_parts = [
    pd.DataFrame({"id": [2], "label": ["B"]}),
    pd.DataFrame({"id": [1], "label": ["A"]}),
]
wrong = pd.concat([
    left.merge(right, on="id", how="inner", validate="1:1")
    for left, right in zip(left_parts, right_parts)
], ignore_index=True)
lookup = pd.concat(right_parts, ignore_index=True)
correct = pd.concat([
    part.merge(lookup, on="id", how="left", validate="m:1")
    for part in left_parts
], ignore_index=True)
print(wrong.shape)  # (0, 2)：同编号的两个块没有匹配项。
print(correct)  # id=1 得到 A，id=2 得到 B；两行两列。

(0, 2)


   id label
0   1     A
1   2     B


### 5.4 避免循环追加整表

不断把“已有结果 + 新块”传给 concat，会反复构造越来越大的结果。需要完整表且总量可放入内存时，可以先收集各块，再一次 concat。

但列表会保留所有块，这种写法仍需容纳全部数据。若目标只是金额均值，就累计两个统计量，不应为了使用 chunksize 又把所有原始块重新拼回整表。

In [13]:
pieces = [pd.DataFrame({"id": [1, 2]}), pd.DataFrame({"id": [3]})]
combined = pd.concat(pieces, ignore_index=True)
print(combined)  # 行标签 0、1、2，id 为 1、2、3。
print(combined.shape)  # (3, 1)；pieces 和 combined 都是当前仍可访问的对象。

   id
0   1
1   2
2   3


(3, 1)


## 6 完整读取与分块统计

### 6.1 明确相同的任务

下面比较同一个任务：只读 amount_cents 列，按 Int64 解析，跳过缺失，返回总额、有效计数和均值。为重复测量，把已经演示的两种读取方式分别放进短函数。以下计时输入至少有一个有效金额；全缺失输入按第 4.3 节另行处理。

完整读取记录整张已读表的内存估计；分块读取记录观察到的最大单块表估计。这两个量都不是进程峰值，后者还没有计入读取器、临时数组及多个对象短暂共存的开销。

In [14]:
def summarize_full(path):
    # 完整读取同一数值列；本节数据至少有一个有效值。
    frame = pd.read_csv(
        path, usecols=["amount_cents"], dtype={"amount_cents": "Int64"},
        engine="c", encoding="utf-8",
    )
    total = int(frame["amount_cents"].sum())
    count = int(frame["amount_cents"].count())
    mean = total / count
    table_bytes = int(frame.memory_usage(deep=True).sum())
    return total, count, mean, table_bytes


def summarize_chunks(path, chunksize=5000):
    # 分块版累计同样的分子和分母，只记录最大的单块内存估计。
    total = 0
    count = 0
    max_chunk_bytes = 0
    with pd.read_csv(
        path, chunksize=chunksize, usecols=["amount_cents"],
        dtype={"amount_cents": "Int64"}, engine="c", encoding="utf-8",
    ) as reader:
        for chunk in reader:
            # 块之间用总和与有效数量合并，缺失值不进入分母。
            total += int(chunk["amount_cents"].sum())
            count += int(chunk["amount_cents"].count())
            chunk_bytes = int(chunk.memory_usage(deep=True).sum())
            max_chunk_bytes = max(max_chunk_bytes, chunk_bytes)
    mean = total / count
    return total, count, mean, max_chunk_bytes

### 6.2 先核对，再记录耗时

实验把 10、20、缺失、缺失、90 的五行模式重复 4,000 次，共 20,000 行；每块最多 5,000 行。手算总额为 480,000 分，有效计数为 12,000，均值为 40 分。先用断言核对两种算法，再计时。

timeit.repeat 重复三轮，每轮执行一次函数。计时包含打开 CSV、读取、类型解析、统计、内存估计与关闭，不包含生成文件和打印。正确性检查已经先读过文件，计时可能受到文件缓存与机器负载影响；输出只记录本次小规模观察，不代表固定加速比。

In [15]:
from timeit import repeat

with TemporaryDirectory() as directory:
    csv_path = Path(directory) / "benchmark.csv"
    # 1. 数据生成和写入放在计时外，两种方法读取同一份 CSV。
    benchmark_data = pd.DataFrame({
        "record_id": range(20000),
        "amount_cents": pd.Series([10, 20, None, None, 90] * 4000, dtype="Int64"),
        "region": ["east", "west"] * 10000,
        "note": ["demo"] * 20000,
    })
    benchmark_data.to_csv(csv_path, index=False, encoding="utf-8")
    del benchmark_data  # 后续统计只从 CSV 读取，不依赖生成表。

    # 2. 先比较任务结果；第四项分别是整表与最大单块的估计字节数。
    full_result = summarize_full(csv_path)
    chunk_result = summarize_chunks(csv_path)
    assert full_result[:3] == chunk_result[:3] == (480000, 12000, 40.0)
    print("总额、有效计数、均值:", full_result[:3])  # 预期：(480000, 12000, 40.0)，金额单位为分。
    print("整表估计字节:", full_result[3])  # 预期：本环境为 180132 字节；这是 DataFrame 估计值，不是进程峰值。
    print("最大单块估计字节:", chunk_result[3])

    # 3. 每次计时包含从 CSV 读取到统计结束，不包含上面的生成和打印。
    full_seconds = repeat(lambda: summarize_full(csv_path), number=1, repeat=3)
    chunk_seconds = repeat(lambda: summarize_chunks(csv_path), number=1, repeat=3)
    print("完整读取秒数:", [round(value, 6) for value in full_seconds])
    print("分块统计秒数:", [round(value, 6) for value in chunk_seconds])
    # 比较完整耗时列表，不预设哪一种更快；内存估计不等于峰值。
print(csv_path.exists())  # False：全部读取完成后清理临时 CSV。

总额、有效计数、均值:

 (480000, 12000, 40.0)
整表估计字节: 180132
最大单块估计字节: 45132


完整读取秒数: [0.017428, 0.016927, 0.016983]
分块统计秒数: [0.031787, 0.033486, 0.033616]
False


## 7 选学：超过内存时的工具入口
先判断瓶颈来自原始数据、最终结果，还是跨块状态。简单的独立转换与可合并汇总可以逐块完成；全局排序或复杂连接可能需要专门的执行方式。

| 工具 | 中文名称／含义与选择条件 |
| --- | --- |
| Dask DataFrame | 分区表格计算；适合继续使用类似 pandas 的操作。全局重排与连接可能产生分区间数据交换，不能假定分区就自动省时。数据缩小到适合内存后可回到 pandas。 |
| DuckDB | 分析型数据库；可用 SQL 表达聚合、连接与排序，并将部分超内存中间数据溢写到磁盘。需要临时磁盘空间，且部分复杂算子组合仍可能内存不足。 |

本节只给选择入口，不安装或运行这些工具。选择后仍须核对其版本、类型与缺失语义、资源限制和完整任务的结果。

## 本章小结

（1）info 与 memory_usage 帮助定位占用较多的列；deep 估计受 dtype 和字符串后端影响，不等于进程峰值。

（2）先用 usecols 少读数据，再根据范围、缺失与精度要求选择类型。category 的收益取决于不同值的数量和数据长度。

（3）分块均值需要累计和与有效计数，不能直接平均块均值。空有效块不贡献计数，最终无有效值时应明确输出规则。

（4）跨块去重、全局排序和连接需要协调；循环 concat 或保留全部块会削弱分块的内存优势。

（5）比较完整读取与分块处理时，先验证相同统计口径，再记录有限规模的实际耗时和内存估计。

## 练习

（1）查看下面表格的 info 和逐列 deep 内存估计。仅保留 units 与 region，把 region 转成 category，再比较转换前后估计。检查值、索引与列顺序，说明为什么该地区列适合尝试分类表示。

In [16]:
exercise_table = pd.DataFrame({
    "units": [1, 2, 3, 4] * 100,
    "region": ["north", "south"] * 200,
    "comment": ["unneeded description"] * 400,
})
# 在此查看、选择与转换，分别打印内存估计。
# 检查：处理后为 (400, 2)，列顺序为 units、region，销量总和仍为 1000。
# 将 region 还原为原类型，检查其值与行索引未改变；不把字节差称为峰值差。

（2）先预测下面两个均值是否相同，再运行核对。解释缺失值存在时，为什么按块行数加权也不能保证正确。

In [17]:
exercise_parts = [pd.Series([10.0, None]), pd.Series([20.0, 30.0, 40.0])]
print(pd.Series([part.mean() for part in exercise_parts]).mean())
print(sum(part.sum() for part in exercise_parts) / sum(part.count() for part in exercise_parts))
# 先写预测，再说明每块均值对应多少条有效观测。
# 在此尝试用块行数加权，与有效计数加权进行比较。

20.0
25.0


（3）将下面 CSV 文本写入临时目录，按每块 2 行累计金额和与有效计数；读写完成后关闭读取器并清理文件。再把输入改成只有两行缺失金额，按本章约定输出缺失均值。

In [18]:
exercise_csv = "id,amount_cents\n1,5\n2,\n3,15\n4,\n5,\n"
exercise_missing_csv = "id,amount_cents\n1,\n2,\n"
# 在此用 TemporaryDirectory、read_csv 的 with 和 chunksize 完成两次统计。
# 指定 amount_cents 为 Int64；原输入的和为 20、有效计数为 2、均值为 10。
# 新输入的有效计数为 0，均值为 pd.NA；不能直接除以零。
# 检查临时路径在退出上下文后不存在。

（4）原任务只要求计算全部记录的金额均值，现增加约束：“同一 id 可能跨块重复，只保留首次出现的记录后再求均值”。判断仅累计各块 sum 和 count 是否仍足够，说明需要新增什么状态；如果不同 id 数量也超过内存，应如何调整方案？用小表验证你的判断。

In [19]:
exercise_blocks = [
    pd.DataFrame({"id": [1, 2], "amount_cents": [10, 20]}),
    pd.DataFrame({"id": [2, 3], "amount_cents": [200, 30]}),
]
# 在此说明方法选择，再对这两个小块实现“保留首次”的结果。
# 检查：最终 id 为 1、2、3，金额 10、20、30，均值 20。
# 解释为什么仅保留每块的和、计数无法撤销重复 id=2 的 200。
# 若使用已见键集合，讨论其大小随输入增长的限制；无需安装新工具。

### 重点练习提示（第 4 题）

提示一：重复 id 可以出现在不同块；判断新记录是否已出现，必须有跨块信息。

提示二：在每块内先保留首次，再排除已见键；随后更新已见键集合和有效值的和、计数。

### 参考解析（第 4 题）

第一块接受 id=1、2，金额为 10、20 分；第二块的 id=2 已见，跳过 200，接受 id=3 的 30 分。最终和为 60 分、有效计数为 3，均值为 20 分。只累计各块的和与计数会得到 260/4=65，丢失键后无法知道应撤销哪一条。已见键集合随不同 id 数增长，不能声称内存始终固定；若它也放不下，需要使用可落盘的键状态或外部排序，并保留来源顺序来定义“首次”。这些是方案选择，本题不要求安装工具或声称已经验证大数据性能。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档或 API 源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | 内存与类型：[Scaling to large datasets](https://pandas.pydata.org/docs/user_guide/scale.html) 的 Load less data、Use efficient datatypes、Use chunking、Use Other Libraries；[DataFrame.info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) 的 memory_usage；[DataFrame.memory_usage](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.memory_usage.html) 的 index、deep 与字节返回值；[String migration](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#brief-introduction-to-the-new-default-string-dtype) 的默认 str 与 PyArrow 存储；[Categorical data — Memory usage](https://pandas.pydata.org/docs/user_guide/categorical.html#memory-usage) 的类别数量边界；[to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) 的 downcast、精度提醒及可空类型示例。读写与统计：[read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) 的 usecols、dtype、engine、chunksize、low_memory；[IO tools — Iterating through files chunk by chunk](https://pandas.pydata.org/docs/user_guide/io.html#iterating-through-files-chunk-by-chunk) 的 TextFileReader 上下文；[to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) 的路径、index、encoding；[Series.sum](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) 的 skipna、min_count 与空／全缺失输入；[count](https://pandas.pydata.org/docs/reference/api/pandas.Series.count.html) 的非缺失计数；[mean](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html) 的缺失处理。跨块语义：[drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) 的 subset、keep；[sort_values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) 的排序；[merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) 的键连接与 validate；[Merging — Concatenating objects](https://pandas.pydata.org/docs/user_guide/merging.html#concatenating-objects) 的避免迭代 concat。 |
| NumPy 官方文档（2.5） | [average](https://numpy.org/doc/2.5/reference/generated/numpy.average.html) 的 weights：加权均值为值与权重乘积之和除以权重总和；本章将权重取为各块有效计数，不需要调用 NumPy。 |
| Python 官方文档（3.12） | [TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory) 的上下文清理；[Path.write_text](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.write_text) 的写入并关闭；[timeit](https://docs.python.org/3.12/library/timeit.html) 的 repeat、number、callable、计时范围与重复测量解释。 |
| Dask 官方文档 | [DataFrames Best Practices](https://docs.dask.org/en/stable/dataframe-best-practices.html) 的 Use Pandas、Reduce, and then use pandas、Avoid Full-Data Shuffling 与 Joins：分区处理的适用条件及跨分区协调成本。 |
| DuckDB 官方文档 | [Tuning Workloads — Larger-than-Memory Workloads](https://duckdb.org/docs/current/guides/performance/how_to_tune_workloads#larger-than-memory-workloads-out-of-core-processing) 的 Spilling to Disk、Blocking Operators、Limitations：磁盘溢写与仍可能内存不足的边界。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[scale](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/scale.rst)、[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)、[categorical](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/categorical.rst)、[io](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/io.rst)、[merging](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/merging.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |